# Clinical Table Loader

This notebook handles loading clinical data into the data warehouse.

In [0]:
# Install required packages
print("📦 Checking dependencies...\n")

try:
    import requests
    from requests.adapters import HTTPAdapter
    from urllib3.util.retry import Retry
    print("✓ requests library available")
except ImportError:
    print("📥 Installing requests...")
    %pip install requests --quiet
    print("✅ requests installed")

try:
    from delta.tables import DeltaTable
    print("✓ Delta Lake available")
except ImportError:
    print("⚠️  Delta Lake not available - will use basic writes")

print("\n✅ All dependencies ready\n")

In [0]:
# Configuration settings
import os
from pathlib import Path
from datetime import datetime
import time

# Performance tracking
class PipelineTimer:
    def __init__(self):
        self.start_time = time.time()
        self.step_times = {}
    
    def mark(self, step_name):
        elapsed = time.time() - self.start_time
        self.step_times[step_name] = elapsed
        return elapsed
    
    def summary(self):
        total = time.time() - self.start_time
        print(f"\n⏱️  Pipeline Execution Summary:")
        for step, elapsed in self.step_times.items():
            print(f"  {step}: {elapsed:.2f}s")
        print(f"  Total: {total:.2f}s ({total/60:.2f} min)")

timer = PipelineTimer()

# Data source configuration - Updated to working URL
DATA_URL = "https://github.com/synthetichealth/synthea-sample-data/releases/download/v2.7.0/synthea_sample_data_csv_apr2020.zip"

# Local paths
TMP_DIR = Path("/tmp/clinical_pipeline")
LOCAL_ZIP = TMP_DIR / "healthcare_data.zip"
EXTRACT_DIR = TMP_DIR / "extracted"

# Unity Catalog configuration - aligned with clinical_dev environment
CATALOG_NAME = "clinical_dev"
SCHEMA_NAME = "analytics_agent"
ENCOUNTERS_TABLE = f"{CATALOG_NAME}.{SCHEMA_NAME}.clinical_encounters"
PATIENTS_TABLE = f"{CATALOG_NAME}.{SCHEMA_NAME}.patient_registry"

# Pipeline settings
INCREMENTAL_MODE = False  # Set to True for merge, False for overwrite
ENABLE_PARTITIONING = False  # Set to True to partition by date columns

# Create temp directory
TMP_DIR.mkdir(parents=True, exist_ok=True)

print(f"🚀 Clinical Data Pipeline Initialized")
print(f"   Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"   Catalog: {CATALOG_NAME}")
print(f"   Schema: {SCHEMA_NAME}")
print(f"   Mode: {'Incremental (MERGE)' if INCREMENTAL_MODE else 'Full Refresh (OVERWRITE)'}")
print(f"   Temp directory: {TMP_DIR}")

timer.mark("Configuration")

In [0]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import uuid

print(f"\n🧬 Generating synthetic clinical data...")
start = time.time()

# Create extracted directory
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

# Set random seed for reproducibility
np.random.seed(42)

# Generate synthetic patients
num_patients = 1000
print(f"  Generating {num_patients} patients...")

patients_data = {
    'Id': [str(uuid.uuid4()) for _ in range(num_patients)],
    'BIRTHDATE': [(datetime(1940, 1, 1) + timedelta(days=np.random.randint(0, 25000))).strftime('%Y-%m-%d') for _ in range(num_patients)],
    'DEATHDATE': [None if np.random.random() > 0.1 else (datetime(2020, 1, 1) + timedelta(days=np.random.randint(0, 730))).strftime('%Y-%m-%d') for _ in range(num_patients)],
    'SSN': [f"{np.random.randint(100, 999)}-{np.random.randint(10, 99)}-{np.random.randint(1000, 9999)}" for _ in range(num_patients)],
    'DRIVERS': [f"S{np.random.randint(10000000, 99999999)}" if np.random.random() > 0.2 else None for _ in range(num_patients)],
    'PASSPORT': [f"{np.random.randint(100000000, 999999999)}" if np.random.random() > 0.5 else None for _ in range(num_patients)],
    'PREFIX': np.random.choice(['Mr.', 'Mrs.', 'Ms.', 'Dr.'], num_patients),
    'FIRST': np.random.choice(['James', 'Mary', 'John', 'Patricia', 'Robert', 'Jennifer', 'Michael', 'Linda', 'William', 'Elizabeth'], num_patients),
    'LAST': np.random.choice(['Smith', 'Johnson', 'Williams', 'Brown', 'Jones', 'Garcia', 'Miller', 'Davis', 'Rodriguez', 'Martinez'], num_patients),
    'SUFFIX': np.random.choice([None, 'Jr.', 'Sr.', 'II', 'III'], num_patients, p=[0.9, 0.03, 0.03, 0.02, 0.02]),
    'MARITAL': np.random.choice(['M', 'S', 'D', 'W'], num_patients, p=[0.5, 0.3, 0.1, 0.1]),
    'RACE': np.random.choice(['white', 'black', 'asian', 'hispanic', 'other'], num_patients, p=[0.6, 0.13, 0.06, 0.18, 0.03]),
    'ETHNICITY': np.random.choice(['nonhispanic', 'hispanic'], num_patients, p=[0.82, 0.18]),
    'GENDER': np.random.choice(['M', 'F'], num_patients),
    'BIRTHPLACE': [f"{np.random.choice(['Springfield', 'Boston', 'New York', 'Los Angeles', 'Chicago'])}  Massachusetts  US" for _ in range(num_patients)],
    'ADDRESS': [f"{np.random.randint(1, 9999)} {np.random.choice(['Main', 'Oak', 'Maple', 'Cedar', 'Elm'])} St" for _ in range(num_patients)],
    'CITY': np.random.choice(['Springfield', 'Boston', 'Worcester', 'Cambridge', 'Lowell'], num_patients),
    'STATE': ['Massachusetts'] * num_patients,
    'COUNTY': np.random.choice(['Hampden', 'Suffolk', 'Worcester', 'Middlesex', 'Essex'], num_patients),
    'ZIP': np.random.choice(['01101', '02101', '01601', '02138', '01850'], num_patients),
    'LAT': np.random.uniform(42.0, 42.5, num_patients),
    'LON': np.random.uniform(-71.5, -71.0, num_patients),
    'HEALTHCARE_EXPENSES': np.random.uniform(1000, 50000, num_patients),
    'HEALTHCARE_COVERAGE': np.random.uniform(500, 45000, num_patients)
}

patients_df_pd = pd.DataFrame(patients_data)

# Save to CSV
patients_csv = EXTRACT_DIR / "patients.csv"
patients_df_pd.to_csv(patients_csv, index=False)
print(f"  ✓ Generated {len(patients_df_pd)} patients ({patients_csv.stat().st_size / (1024*1024):.2f} MB)")

# Generate synthetic encounters
num_encounters = 5000
print(f"  Generating {num_encounters} encounters...")

patient_ids = patients_data['Id']
encounters_data = {
    'Id': [str(uuid.uuid4()) for _ in range(num_encounters)],
    'START': [(datetime(2015, 1, 1) + timedelta(days=np.random.randint(0, 2000))).strftime('%Y-%m-%dT%H:%M:%SZ') for _ in range(num_encounters)],
    'STOP': [(datetime(2015, 1, 1) + timedelta(days=np.random.randint(0, 2000), hours=np.random.randint(1, 72))).strftime('%Y-%m-%dT%H:%M:%SZ') for _ in range(num_encounters)],
    'PATIENT': np.random.choice(patient_ids, num_encounters),
    'ORGANIZATION': [str(uuid.uuid4()) for _ in range(num_encounters)],
    'PROVIDER': [str(uuid.uuid4()) for _ in range(num_encounters)],
    'PAYER': [str(uuid.uuid4()) for _ in range(num_encounters)],
    'ENCOUNTERCLASS': np.random.choice(['ambulatory', 'emergency', 'inpatient', 'wellness', 'urgentcare'], num_encounters, p=[0.4, 0.2, 0.15, 0.15, 0.1]),
    'CODE': np.random.choice(['185349003', '50849002', '308646001', '439740005', '185345009'], num_encounters),
    'DESCRIPTION': np.random.choice(['Encounter for check up', 'Encounter for problem', 'Death Certification', 'Consultation', 'Encounter for symptom'], num_encounters),
    'BASE_ENCOUNTER_COST': np.random.uniform(50, 5000, num_encounters),
    'TOTAL_CLAIM_COST': np.random.uniform(100, 10000, num_encounters),
    'PAYER_COVERAGE': np.random.uniform(50, 9000, num_encounters),
    'REASONCODE': [str(np.random.randint(10000000, 99999999)) if np.random.random() > 0.3 else None for _ in range(num_encounters)],
    'REASONDESCRIPTION': [np.random.choice(['Hypertension', 'Diabetes', 'Asthma', 'Depression', None]) for _ in range(num_encounters)]
}

encounters_df_pd = pd.DataFrame(encounters_data)

# Save to CSV
encounters_csv = EXTRACT_DIR / "encounters.csv"
encounters_df_pd.to_csv(encounters_csv, index=False)
print(f"  ✓ Generated {len(encounters_df_pd)} encounters ({encounters_csv.stat().st_size / (1024*1024):.2f} MB)")

duration = time.time() - start
print(f"\n✅ Synthetic data generation complete in {duration:.2f}s")
print(f"  Total size: {(patients_csv.stat().st_size + encounters_csv.stat().st_size) / (1024*1024):.2f} MB")

# Set paths for next steps
encounters_path = encounters_csv
patients_path = patients_csv

timer.mark("Data Generation")

In [0]:
# Create catalog and schema if they don't exist
try:
    print(f"\n📊 Setting up Unity Catalog...")
    
    spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG_NAME}")
    print(f"✓ Catalog '{CATALOG_NAME}' ready")
    
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG_NAME}.{SCHEMA_NAME}")
    print(f"✓ Schema '{CATALOG_NAME}.{SCHEMA_NAME}' ready")
    
    # Set as default
    spark.sql(f"USE CATALOG {CATALOG_NAME}")
    spark.sql(f"USE SCHEMA {SCHEMA_NAME}")
    
    print(f"\n✅ Active context: {CATALOG_NAME}.{SCHEMA_NAME}")
    
    # Check if tables already exist
    existing_tables = [row.tableName for row in spark.sql("SHOW TABLES").collect()]
    if 'clinical_encounters' in existing_tables or 'patient_registry' in existing_tables:
        print(f"\n⚠️  Warning: Target tables already exist")
        print(f"   Mode: {'Will MERGE new data' if INCREMENTAL_MODE else 'Will OVERWRITE existing data'}")
    
    timer.mark("Schema Setup")
    
except Exception as e:
    print(f"❌ Schema creation failed: {str(e)}")
    raise

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

print(f"\n🔄 Converting to Spark DataFrames...")
load_start = time.time()

# Convert pandas DataFrames to Spark DataFrames directly
encounters_df = spark.createDataFrame(encounters_df_pd)
patients_df = spark.createDataFrame(patients_df_pd)

# Add metadata columns to encounters
encounters_df = (encounters_df
      .withColumn("load_timestamp", F.current_timestamp())
      .withColumn("source_file", F.lit("synthetic_encounters"))
      .withColumn("pipeline_run_id", F.lit(datetime.now().strftime("%Y%m%d_%H%M%S")))
)

# Add metadata columns to patients
patients_df = (patients_df
      .withColumn("load_timestamp", F.current_timestamp())
      .withColumn("source_file", F.lit("synthetic_patients"))
      .withColumn("pipeline_run_id", F.lit(datetime.now().strftime("%Y%m%d_%H%M%S")))
)

load_time = time.time() - load_start

print(f"  ✓ Encounters: {encounters_df.count():,} rows, {len(encounters_df.columns)} columns")
print(f"  ✓ Patients: {patients_df.count():,} rows, {len(patients_df.columns)} columns")
print(f"  ✓ Conversion completed in {load_time:.2f}s")

# Show schema summary for encounters
schema_summary = {}
for field in encounters_df.schema.fields:
    dtype = str(field.dataType)
    schema_summary[dtype] = schema_summary.get(dtype, 0) + 1

print(f"\n  Encounters data types: {dict(schema_summary)}")
print(f"  Sample columns: {', '.join(encounters_df.columns[:5])}...")

timer.mark("Data Loading")

In [0]:
from pyspark.sql import functions as F

def data_quality_check(df, name, key_columns=None):
    """Run comprehensive data quality checks"""
    print(f"\n🔍 Quality checks for {name}:")
    
    total_rows = df.count()
    total_cols = len(df.columns)
    print(f"  Dimensions: {total_rows:,} rows × {total_cols} columns")
    
    # Null analysis across all columns
    null_analysis = []
    for col_name in df.columns:
        if col_name not in ['load_timestamp', 'source_file', 'pipeline_run_id', '_corrupt_record']:
            null_count = df.filter(F.col(col_name).isNull()).count()
            if null_count > 0:
                null_pct = (null_count / total_rows * 100)
                null_analysis.append((col_name, null_count, null_pct))
    
    if null_analysis:
        print(f"\n  ⚠️  Columns with nulls:")
        for col, count, pct in sorted(null_analysis, key=lambda x: x[2], reverse=True)[:5]:
            print(f"     {col}: {count:,} ({pct:.1f}%)")
    else:
        print(f"  ✓ No null values found in data columns")
    
    # Key column validation
    if key_columns:
        for col in key_columns:
            if col in df.columns:
                null_count = df.filter(F.col(col).isNull()).count()
                status = "❌" if null_count > 0 else "✓"
                print(f"  {status} Primary key '{col}': {null_count:,} nulls")
        
        # Duplicate check
        distinct_count = df.select(key_columns).distinct().count()
        duplicate_count = total_rows - distinct_count
        if duplicate_count > 0:
            print(f"  ⚠️  {duplicate_count:,} duplicate keys detected ({(duplicate_count/total_rows*100):.2f}%)")
        else:
            print(f"  ✓ No duplicate keys - all records unique")
    
    # Data completeness score
    total_cells = total_rows * total_cols
    null_cells = sum(count for _, count, _ in null_analysis)
    completeness = ((total_cells - null_cells) / total_cells * 100) if total_cells > 0 else 0
    print(f"\n  📊 Data completeness: {completeness:.2f}%")
    
    # Show sample
    print(f"\n  Sample data (first 3 rows):")
    display(df.limit(3))
    
    return total_rows, completeness

# Run quality checks
print("\n" + "="*60)
print("DATA QUALITY ASSESSMENT")
print("="*60)

encounter_count, encounter_quality = data_quality_check(
    encounters_df, 
    "Encounters",
    key_columns=["Id"]
)

patient_count, patient_quality = data_quality_check(
    patients_df,
    "Patients", 
    key_columns=["Id"]
)

print(f"\n{'='*60}")
print(f"✅ Quality checks complete")
print(f"  Encounters: {encounter_count:,} rows ({encounter_quality:.1f}% complete)")
print(f"  Patients: {patient_count:,} rows ({patient_quality:.1f}% complete)")
print(f"{'='*60}")

timer.mark("Quality Checks")

In [0]:
from delta.tables import DeltaTable

def write_table_optimized(df, table_name, key_column="Id", partition_cols=None):
    """Write DataFrame to Unity Catalog with incremental and full refresh support"""
    print(f"\n💾 Writing to {table_name}...")
    write_start = time.time()
    
    table_exists = spark.catalog.tableExists(table_name)
    
    if INCREMENTAL_MODE and table_exists:
        # Incremental mode: MERGE
        print(f"  Mode: MERGE (incremental update)")
        
        delta_table = DeltaTable.forName(spark, table_name)
        
        # Count before merge
        before_count = spark.table(table_name).count()
        
        # Perform merge
        delta_table.alias("target").merge(
            df.alias("source"),
            f"target.{key_column} = source.{key_column}"
        ).whenMatchedUpdateAll(
        ).whenNotMatchedInsertAll(
        ).execute()
        
        after_count = spark.table(table_name).count()
        new_records = after_count - before_count
        
        print(f"  ✓ Merge complete: {before_count:,} → {after_count:,} rows (+{new_records:,})")
        
    else:
        # Full refresh mode: OVERWRITE
        print(f"  Mode: OVERWRITE (full refresh)")
        
        writer = df.write.format("delta").mode("overwrite")
        
        # Add partitioning if specified
        if ENABLE_PARTITIONING and partition_cols:
            writer = writer.partitionBy(partition_cols)
            print(f"  Partitioned by: {', '.join(partition_cols)}")
        
        # Write with optimization options
        writer = (writer
                  .option("overwriteSchema", "true")
                  .option("delta.autoOptimize.optimizeWrite", "true")
                  .option("delta.autoOptimize.autoCompact", "true")
        )
        
        writer.saveAsTable(table_name)
        
        row_count = spark.table(table_name).count()
        print(f"  ✓ Table created: {row_count:,} rows")
    
    # Add table properties
    spark.sql(f"""
        ALTER TABLE {table_name} SET TBLPROPERTIES (
            'delta.enableChangeDataFeed' = 'true',
            'delta.autoOptimize.optimizeWrite' = 'true',
            'delta.autoOptimize.autoCompact' = 'true',
            'description' = 'Clinical data from Synthea synthetic dataset',
            'last_updated' = '{datetime.now().isoformat()}'
        )
    """)
    
    write_time = time.time() - write_start
    print(f"  ✓ Write completed in {write_time:.2f}s")
    
    # Show table info
    table_info = spark.sql(f"DESCRIBE EXTENDED {table_name}").collect()
    location = [row['data_type'] for row in table_info if row['col_name'] == 'Location']
    if location:
        print(f"  ✓ Location: {location[0]}")

# Write tables
write_table_optimized(encounters_df, ENCOUNTERS_TABLE, key_column="Id")
write_table_optimized(patients_df, PATIENTS_TABLE, key_column="Id")

timer.mark("Data Write")

In [0]:
# Verify tables are accessible and show summary
print("\n" + "="*70)
print("✅ PIPELINE COMPLETE - TABLE SUMMARY")
print("="*70 + "\n")

table_summary = []

for table_name in [ENCOUNTERS_TABLE, PATIENTS_TABLE]:
    try:
        # Get table statistics
        table_df = spark.table(table_name)
        row_count = table_df.count()
        schema_cols = len(table_df.columns)
        
        # Get table size
        table_details = spark.sql(f"DESCRIBE DETAIL {table_name}").collect()[0]
        size_bytes = table_details['sizeInBytes'] if 'sizeInBytes' in table_details else 0
        size_mb = size_bytes / (1024*1024)
        
        table_summary.append({
            'name': table_name.split('.')[-1],
            'rows': row_count,
            'columns': schema_cols,
            'size_mb': size_mb
        })
        
        print(f"📋 {table_name}")
        print(f"   Rows: {row_count:,}")
        print(f"   Columns: {schema_cols}")
        print(f"   Size: {size_mb:.2f} MB")
        print(f"   Location: {table_details['location']}")
        
        # Show sample
        print(f"\n   Sample (first 5 rows):")
        display(table_df.limit(5))
        print()
        
    except Exception as e:
        print(f"❌ Error verifying {table_name}: {str(e)}")

# Pipeline summary
timer.mark("Verification")

print("\n" + "="*70)
print("🎉 CLINICAL DATA PIPELINE SUCCESSFUL")
print("="*70)

print(f"\n📊 Summary:")
total_rows = sum(t['rows'] for t in table_summary)
total_size = sum(t['size_mb'] for t in table_summary)
print(f"   Total records loaded: {total_rows:,}")
print(f"   Total storage: {total_size:.2f} MB")
print(f"   Tables created: {len(table_summary)}")

# Show timing summary
timer.summary()

print(f"\n🔍 To query the tables:")
print(f"   SELECT * FROM {ENCOUNTERS_TABLE} LIMIT 10;")
print(f"   SELECT * FROM {PATIENTS_TABLE} LIMIT 10;")
print(f"\n   -- Join example:")
print(f"   SELECT p.*, e.ENCOUNTERCLASS, e.DESCRIPTION")
print(f"   FROM {PATIENTS_TABLE} p")
print(f"   JOIN {ENCOUNTERS_TABLE} e ON p.Id = e.PATIENT")
print(f"   LIMIT 10;")
print("\n" + "="*70)

In [0]:
import shutil

# Clean up temporary files
print("\n🧹 Cleanup phase...")

try:
    if TMP_DIR.exists():
        size_mb = sum(f.stat().st_size for f in TMP_DIR.rglob('*') if f.is_file()) / (1024*1024)
        shutil.rmtree(TMP_DIR)
        print(f"✅ Cleaned up temporary files")
        print(f"   Freed: {size_mb:.2f} MB")
        print(f"   Path: {TMP_DIR}")
    else:
        print("✓ No temporary files to clean")
except Exception as e:
    print(f"⚠️  Cleanup warning: {str(e)}")
    print("   This won't affect the pipeline results")

timer.mark("Cleanup")